# Stage 2: Best Baseline + CatBoost Residual

Чистовой ноутбук для текущей лучшей ветки решений.

Что внутри:

1. Точная best baseline эвристика команды.
2. CatBoost residual поверх этой эвристики.
3. Генерация confirmed candidates `14` и `15`.
4. Заготовка для производных кандидатов по сетке `alpha/clip`.

Известные contest-результаты:

| candidate | description | contest MAPE |
|---|---|---:|
| best baseline / `test_best.csv` | global seasonal factor + conservative momentum | ~5.72 |
| `candidate_14` | mild CatBoost residual, `alpha=0.03`, clip ±2% | 5.707 |
| `candidate_15` | aggressive CatBoost residual, `alpha=0.50`, no clip | 5.58 |

Главный вывод: ML residual поверх правильного baseline уже подтвердился на leaderboard.

## 1. Best Baseline Formula

Baseline не использует индивидуальный коэффициент `март / февраль` конкретного магазина. Индивидуальные коэффициенты оказались шумными.

Вместо этого используется глобальный медианный сезонный коэффициент по двум известным мартовским переходам:

`final_factor = median(РТО_март_2023 / РТО_февраль_2023, РТО_март_2024 / РТО_февраль_2024)`

Для текущих данных:

`final_factor = 1.119179025171653`

Momentum:

`momentum = РТО_февраль_2025 / mean(РТО_декабрь_2024, РТО_январь_2025, РТО_февраль_2025)`

Затем momentum ограничивается и применяется очень мягко:

`momentum_correction = clip(momentum, 0.95, 1.05) ** 0.25`

Итоговый baseline:

`base = РТО_февраль_2025 * final_factor * momentum_correction`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TRAIN_PATH = ROOT / 'data' / 'train_2.csv'
SUBMISSIONS_DIR = ROOT / 'submissions'
RESULTS_DIR = ROOT / 'results'
TARGET = 'РТО'
SEED = 2026

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)
ROOT

PosixPath('/home/banserv/Gradient Rosta 2026')

In [2]:
cat_cols = [
    'Дата открытия, категориальный',
    'Торговая площадь, категориальный',
    'Населенный пункт',
    'Регион',
]

static_num_cols = [
    'Численность населения', 'Количество домохозяйств', 'Трафик пеший, в час', 'Трафик авто, в час',
    'Маркетплейсы, доставки, постаматы (100 м)', 'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)',
    'Остановки (300 м)', 'Продуктовые магазины (500 м)', 'Пятерочки (500 м)', 'Количество касс',
    'Флаг алкогольной лицензии',
]

behavior_cols = [
    'Среднее количество промо товаров в чеке',
    'Среднее количество товаров в чеке',
    'Среднее количество отмен',
    'Рабочие часы в день',
]

def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(y_true == 0, 1.0, y_true)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100)

In [ ]:
# Plot РТО by month
rto_by_month = rto.mean(axis=0)
rto_by_month.plot(figsize=(12, 6), marker='o')
plt.title('Среднее РТО по месяцам')
plt.xlabel('Период')
plt.ylabel('РТО')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Compute Exact Baseline Factor

Считаем фактор так же, как в командном репозитории `Growth-Gradient`.

In [3]:
df = pd.read_csv(TRAIN_PATH, low_memory=False)

period_label = pd.to_datetime(
    df['Год'].astype(str) + '-' + df['Месяц'].astype(str).str.zfill(2) + '-01'
).dt.strftime('%Y-%m')

rto = df.assign(period_label=period_label).pivot_table(
    index='new_id', columns='period_label', values=TARGET, aggfunc='mean'
)

ratio_2023 = (rto['2023-03'] / rto['2023-02']).replace([np.inf, -np.inf], np.nan)
ratio_2024 = (rto['2024-03'] / rto['2024-02']).replace([np.inf, -np.inf], np.nan)

validation_factor = ratio_2023.median()
final_factor = pd.concat([ratio_2023, ratio_2024]).dropna().median()

validation_factor, final_factor

(np.float64(1.1294587539218028), np.float64(1.119179025171653))

## 3. Reproduce Best Baseline

Этот блок воспроизводит `test_best.csv`. Если файл есть в корне проекта, проверяем совпадение.

In [4]:
feb_2025 = rto['2025-02']
rolling_3_2025 = (rto['2024-12'] + rto['2025-01'] + rto['2025-02']) / 3
momentum_2025 = (feb_2025 / rolling_3_2025).replace([np.inf, -np.inf], np.nan).fillna(1).clip(0.95, 1.05)
best_baseline_pred = feb_2025 * final_factor * (momentum_2025 ** 0.25)

best_baseline_submission = best_baseline_pred.reset_index()
best_baseline_submission.columns = ['new_id', 'rto']
best_baseline_submission['new_id'] = best_baseline_submission['new_id'].astype(int)
best_baseline_submission['rto'] = best_baseline_submission['rto'].round(2)
best_baseline_submission = best_baseline_submission.sort_values('new_id').reset_index(drop=True)

out_path = SUBMISSIONS_DIR / 'stage2_candidate_13_global_median_seasonal_momentum.csv'
best_baseline_submission.to_csv(out_path, index=False)

if (ROOT / 'test_best.csv').exists():
    test_best = pd.read_csv(ROOT / 'test_best.csv').sort_values('new_id').reset_index(drop=True)
    print('matches test_best:', best_baseline_submission.equals(test_best))
    print('max_abs_diff:', (best_baseline_submission['rto'] - test_best['rto']).abs().max())

out_path, best_baseline_submission.shape, best_baseline_submission.head()

matches test_best: True
max_abs_diff: 0.0


(PosixPath('/home/banserv/Gradient Rosta 2026/submissions/stage2_candidate_13_global_median_seasonal_momentum.csv'),
 (18657, 2),
    new_id          rto
 0       0  91332389.98
 1       1  44116082.01
 2       2  90281968.20
 3       3  66134000.97
 4       4  81598205.46)

## 4. Feature Engineering For Residual Model

CatBoost учит не РТО напрямую, а residual относительно best baseline:

`target = log(РТО_март_2024 / base_март_2024)`

Для мартового train 2024 используем `validation_factor = median(март_2023 / февраль_2023)`, потому что при прогнозе марта 2024 будущий переход `март_2024 / февраль_2024` неизвестен.

Для финального марта 2025 используем `final_factor`, посчитанный по двум известным переходам 2023 и 2024.

In [5]:
def make_panel(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    out['period'] = out['Год'] * 12 + out['Месяц']
    out = out.sort_values(['new_id', 'period']).reset_index(drop=True)

    for c in cat_cols:
        out[c] = out[c].astype(str)

    g = out.groupby('new_id')[TARGET]
    for k in [1, 2, 3, 4, 5, 6, 12, 13]:
        out[f'lag{k}'] = g.shift(k)

    out['rolling3_mean'] = out[['lag1', 'lag2', 'lag3']].mean(axis=1)
    out['momentum_raw'] = out['lag1'] / out['rolling3_mean']
    out['momentum_clip'] = out['momentum_raw'].clip(0.95, 1.05)
    out['momentum_corr'] = out['momentum_clip'] ** 0.25

    out['store_mar_feb_ratio'] = out['lag12'] / out['lag13']
    out['delta_yoy'] = out['lag12'] + (out['lag1'] - out['lag13'])
    out['trend_yoy'] = out['lag12'] * (out['lag1'] / out['lag13'])

    out['diff1'] = out['lag1'] - out['lag2']
    out['diff2'] = out['lag2'] - out['lag3']
    out['accel1'] = out['diff1'] - out['diff2']
    out['lag1_to_lag12'] = out['lag1'] / out['lag12']
    out['lag1_to_lag13'] = out['lag1'] / out['lag13']

    s1 = g.shift(1)
    for w in [3, 6, 12]:
        rolled = s1.groupby(out['new_id']).rolling(w, min_periods=2)
        out[f'roll{w}_mean'] = rolled.mean().reset_index(level=0, drop=True)
        out[f'roll{w}_std'] = rolled.std().reset_index(level=0, drop=True)
        out[f'roll{w}_min'] = rolled.min().reset_index(level=0, drop=True)
        out[f'roll{w}_max'] = rolled.max().reset_index(level=0, drop=True)

    out['volatility_rel3'] = out['roll3_std'] / out['roll3_mean']
    out['volatility_rel6'] = out['roll6_std'] / out['roll6_mean']

    for c in behavior_cols:
        bg = out.groupby('new_id')[c]
        out[f'{c}_lag1'] = bg.shift(1)
        out[f'{c}_lag2'] = bg.shift(2)
        out[f'{c}_lag3'] = bg.shift(3)
        out[f'{c}_lag12'] = bg.shift(12)
        out[f'{c}_lag13'] = bg.shift(13)
        out[f'{c}_diff1'] = out[f'{c}_lag1'] - out[f'{c}_lag2']
        out[f'{c}_yoy_shift'] = out[f'{c}_lag1'] - out[f'{c}_lag13']

    return out.replace([np.inf, -np.inf], np.nan)

def add_best_base(frame: pd.DataFrame, factor: float) -> pd.DataFrame:
    out = frame.copy()
    out['best_base'] = out['lag1'] * factor * out['momentum_corr']
    return out

In [6]:
feature_cols = [
    'lag1', 'lag2', 'lag3', 'lag4', 'lag5', 'lag6', 'lag12', 'lag13',
    'rolling3_mean', 'momentum_raw', 'momentum_clip', 'momentum_corr', 'store_mar_feb_ratio',
    'delta_yoy', 'trend_yoy', 'diff1', 'diff2', 'accel1', 'lag1_to_lag12', 'lag1_to_lag13',
    'roll3_mean', 'roll3_std', 'roll3_min', 'roll3_max',
    'roll6_mean', 'roll6_std', 'roll6_min', 'roll6_max',
    'roll12_mean', 'roll12_std', 'roll12_min', 'roll12_max',
    'volatility_rel3', 'volatility_rel6',
] + static_num_cols + cat_cols

for c in behavior_cols:
    feature_cols += [f'{c}_lag1', f'{c}_lag2', f'{c}_lag3', f'{c}_lag12', f'{c}_lag13', f'{c}_diff1', f'{c}_yoy_shift']

cat_feature_idx = [feature_cols.index(c) for c in cat_cols]

panel = add_best_base(make_panel(df), validation_factor)
train_march = panel[
    (panel['Год'] == 2024) &
    (panel['Месяц'] == 3) &
    (panel[TARGET] > 0) &
    (panel['best_base'] > 0)
].copy()
train_march['target_log_ratio'] = np.log(train_march[TARGET] / train_march['best_base'])

train_march.shape, len(feature_cols)

((18657, 89), 78)

## 5. Cross-Sectional CV On March 2024

У нас есть только один честный мартовый target для residual (`2024-03`), поэтому делаем cross-sectional split по `new_id % 5`. Это не идеальный proxy, но он показал, что residual-модель имеет сильный сигнал.

Зафиксированный результат прогона:

| model | MAPE |
|---|---:|
| clean best baseline | 4.1280 |
| CatBoost residual, `alpha=0.03`, clip ±2% | ~4.09 |
| CatBoost residual, `alpha=0.50`, no clip | 3.6498 |

На leaderboard `alpha=0.03` дал `5.707`, а `alpha=0.50` дал `5.58`.

In [7]:
params = dict(
    loss_function='MAE',
    eval_metric='MAE',
    iterations=250,
    learning_rate=0.04,
    depth=5,
    l2_leaf_reg=12.0,
    random_seed=SEED,
    allow_writing_files=False,
    verbose=False,
)

alpha_grid = [0, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50]
clip_grid = [0.01, 0.02, 0.03, 0.05, 0.08, None]
rows = []

for fold in range(5):
    valid_mask = (train_march['new_id'] % 5) == fold
    tr = train_march[~valid_mask].copy()
    va = train_march[valid_mask].copy()

    model = CatBoostRegressor(**params)
    model.fit(Pool(tr[feature_cols], tr['target_log_ratio'], cat_features=cat_feature_idx))

    pred_log = model.predict(va[feature_cols])
    y = va[TARGET].to_numpy(dtype=float)
    base = va['best_base'].to_numpy(dtype=float)

    for alpha in alpha_grid:
        raw = base * np.exp(alpha * pred_log)
        for clip in clip_grid:
            if clip is None:
                pred = raw
                clip_name = 'no_clip'
            else:
                pred = base * np.clip(raw / base, 1 - clip, 1 + clip)
                clip_name = f'clip_{clip:.2f}'

            rows.append({
                'fold': fold,
                'alpha': alpha,
                'clip': clip_name,
                'mape_ml': mape(y, pred),
                'mape_base': mape(y, base),
                'avg_abs_corr_pct': float(np.mean(np.abs(pred / base - 1)) * 100),
            })

cv = pd.DataFrame(rows)
cv_summary = cv.groupby(['alpha', 'clip'], as_index=False).agg(
    mape_ml=('mape_ml', 'mean'),
    mape_base=('mape_base', 'mean'),
    avg_abs_corr_pct=('avg_abs_corr_pct', 'mean'),
)
cv_summary['gain_vs_base'] = cv_summary['mape_base'] - cv_summary['mape_ml']
cv_summary = cv_summary.sort_values('mape_ml').reset_index(drop=True)

cv.to_csv(RESULTS_DIR / 'best_baseline_catboost_march2024_cv.csv', index=False)
cv_summary.to_csv(RESULTS_DIR / 'best_baseline_catboost_march2024_cv_summary.csv', index=False)
cv_summary.head(20)

,alpha,clip,mape_ml,mape_base,avg_abs_corr_pct,gain_vs_base
0,0.50,no_clip,3.649778,4.127973,1.043734,0.478195
1,0.50,clip_0.08,3.725106,4.127973,0.865429,0.402867
2,0.50,clip_0.05,3.735703,4.127973,0.839763,0.392270
3,0.50,clip_0.03,3.744026,4.127973,0.819172,0.383948
4,0.50,clip_0.02,3.758151,4.127973,0.793028,0.369822
5,0.30,no_clip,3.819496,4.127973,0.618173,0.308477
6,0.50,clip_0.01,3.836708,4.127973,0.647697,0.291266
7,0.30,clip_0.08,3.851629,4.127973,0.544356,0.276344
8,0.30,clip_0.05,3.861975,4.127973,0.521569,0.265998
9,0.30,clip_0.03,3.869177,4.127973,0.504576,0.258796


## 6. Train Final Model And Generate Candidates

Обучаем финальную модель на всех строках марта 2024 и прогнозируем март 2025.

Подтвержденные candidates:

- `candidate_14`: `alpha=0.03`, clip ±2%, leaderboard `5.707`;
- `candidate_15`: `alpha=0.50`, no clip, leaderboard `5.58`.

In [8]:
final_model = CatBoostRegressor(**params)
final_model.fit(Pool(train_march[feature_cols], train_march['target_log_ratio'], cat_features=cat_feature_idx))

feb_rows = df[(df['Год'] == 2025) & (df['Месяц'] == 2)].copy()
mar_rows = feb_rows.copy()
mar_rows['Год'] = 2025
mar_rows['Месяц'] = 3
mar_rows[TARGET] = np.nan

pred_panel = add_best_base(make_panel(pd.concat([df, mar_rows], ignore_index=True)), final_factor)
pred_rows = pred_panel[(pred_panel['Год'] == 2025) & (pred_panel['Месяц'] == 3)].copy()

pred_log = final_model.predict(pred_rows[feature_cols])
base = pred_rows['best_base'].to_numpy(dtype=float)

def make_submission(filename: str, alpha: float, clip: float | None):
    raw = base * np.exp(alpha * pred_log)
    if clip is None:
        pred = raw
    else:
        pred = base * np.clip(raw / base, 1 - clip, 1 + clip)
    pred = np.maximum(pred, 0.0)

    submission = pred_rows[['new_id']].copy()
    submission['rto'] = np.round(pred, 2)
    submission = submission.sort_values('new_id').reset_index(drop=True)
    out_path = SUBMISSIONS_DIR / filename
    submission.to_csv(out_path, index=False)

    ratio = pred / base
    return {
        'file': filename,
        'alpha': alpha,
        'clip': clip,
        'rows': len(submission),
        'mean_abs_correction_pct': float(np.mean(np.abs(ratio - 1)) * 100),
        'p99_correction': float(pd.Series(ratio).quantile(0.99)),
        'max_correction': float(pd.Series(ratio).max()),
    }

confirmed = [
    make_submission('stage2_candidate_14_best_baseline_catboost_residual_conservative.csv', 0.03, 0.02),
    make_submission('stage2_candidate_15_best_baseline_catboost_residual_cvbest.csv', 0.50, None),
]
pd.DataFrame(confirmed)

,file,alpha,clip,rows,mean_abs_correction_pct,p99_correction,max_correction
0,stage2_candidate_14_best_baseline_catboost_res...,0.03,0.02,18657,0.057378,1.004285,1.020000
1,stage2_candidate_15_best_baseline_catboost_res...,0.50,NaN,18657,1.001264,1.073866,1.495312


## 7. Derivative Candidate Grid

Так как `candidate_15` улучшил leaderboard до `5.58`, имеет смысл тестировать производные вокруг него.

Этот блок генерирует дополнительные candidates, но не перезаписывает подтвержденные файлы. Рекомендуемый порядок отправки:

1. `alpha=0.60`, no clip;
2. `alpha=0.70`, no clip;
3. `alpha=0.50`, clip ±10%;
4. `alpha=0.70`, clip ±10%;
5. `alpha=0.80`, no clip.

In [9]:
grid_specs = [
    ('stage2_candidate_16_best_cb_alpha_040_noclip.csv', 0.40, None),
    ('stage2_candidate_17_best_cb_alpha_060_noclip.csv', 0.60, None),
    ('stage2_candidate_18_best_cb_alpha_070_noclip.csv', 0.70, None),
    ('stage2_candidate_19_best_cb_alpha_080_noclip.csv', 0.80, None),
    ('stage2_candidate_20_best_cb_alpha_050_clip_010.csv', 0.50, 0.10),
    ('stage2_candidate_21_best_cb_alpha_070_clip_010.csv', 0.70, 0.10),
    ('stage2_candidate_22_best_cb_alpha_050_clip_005.csv', 0.50, 0.05),
    ('stage2_candidate_23_best_cb_alpha_070_clip_015.csv', 0.70, 0.15),
]

grid_rows = [make_submission(filename, alpha, clip) for filename, alpha, clip in grid_specs]
grid_report = pd.DataFrame(grid_rows)
grid_report.to_csv(RESULTS_DIR / 'best_baseline_catboost_derivative_candidates.csv', index=False)
grid_report

,file,alpha,clip,rows,mean_abs_correction_pct,p99_correction,max_correction
0,stage2_candidate_16_best_cb_alpha_040_noclip.csv,0.4,NaN,18657,0.794179,1.058669,1.379702
1,stage2_candidate_17_best_cb_alpha_060_noclip.csv,0.6,NaN,18657,1.212216,1.089282,1.620608
2,stage2_candidate_18_best_cb_alpha_070_noclip.csv,0.7,NaN,18657,1.427285,1.104918,1.756404
3,stage2_candidate_19_best_cb_alpha_080_noclip.csv,0.8,NaN,18657,1.646739,1.120780,1.903578
4,stage2_candidate_20_best_cb_alpha_050_clip_010...,0.5,0.10,18657,0.813345,1.073866,1.100000
5,stage2_candidate_21_best_cb_alpha_070_clip_010...,0.7,0.10,18657,1.099775,1.100000,1.100000
6,stage2_candidate_22_best_cb_alpha_050_clip_005...,0.5,0.05,18657,0.762571,1.050000,1.050000
7,stage2_candidate_23_best_cb_alpha_070_clip_015...,0.7,0.15,18657,1.148325,1.104918,1.150000


## 8. Feature Importance

На текущем прогоне самые важные признаки были `momentum_raw`, `volatility_rel3`, `volatility_rel6`, `diff1`, `lag1_to_lag12`, `momentum_clip`. Это подтверждает, что модель учит корректировать baseline через краткосрочную динамику и волатильность.

In [10]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.get_feature_importance(
        Pool(train_march[feature_cols], train_march['target_log_ratio'], cat_features=cat_feature_idx),
        type='FeatureImportance',
    ),
}).sort_values('importance', ascending=False)
feature_importance.to_csv(RESULTS_DIR / 'best_baseline_catboost_feature_importance.csv', index=False)
feature_importance.head(20)

,feature,importance
9,momentum_raw,28.710042
32,volatility_rel3,18.787371
33,volatility_rel6,9.856618
15,diff1,8.615568
18,lag1_to_lag12,4.717089
10,momentum_clip,4.713984
35,Количество домохозяйств,2.482055
17,accel1,2.161438
42,Продуктовые магазины (500 м),1.884654
11,momentum_corr,1.760101


## 9. SHAP Values

CatBoost умеет считать SHAP values напрямую. Здесь SHAP считается для residual-модели, то есть объясняет не сам `РТО`, а поправку к best baseline:

`model_pred = log(РТО / best_base)`

Интерпретация:

- положительный SHAP двигает прогноз выше baseline;
- отрицательный SHAP двигает прогноз ниже baseline;
- `mean_abs_shap` показывает, какие признаки сильнее всего влияют на residual.

Чтобы ноутбук не был слишком тяжелым, SHAP считается на фиксированной выборке строк марта 2024. При необходимости `SHAP_SAMPLE_SIZE` можно увеличить.

In [11]:
SHAP_SAMPLE_SIZE = 3000

shap_sample = train_march.sample(
    n=min(SHAP_SAMPLE_SIZE, len(train_march)),
    random_state=SEED,
)
shap_pool = Pool(shap_sample[feature_cols], shap_sample['target_log_ratio'], cat_features=cat_feature_idx)

# Последняя колонка в CatBoost ShapValues — expected value, ее исключаем из per-feature summary.
shap_values = final_model.get_feature_importance(shap_pool, type='ShapValues')
feature_shap = shap_values[:, :-1]

shap_summary = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': np.abs(feature_shap).mean(axis=0),
    'mean_shap': feature_shap.mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

shap_summary.to_csv(RESULTS_DIR / 'best_baseline_catboost_shap_summary.csv', index=False)
shap_summary.head(25)

,feature,mean_abs_shap,mean_shap
0,momentum_raw,0.005064,-0.000020
1,Количество домохозяйств,0.003388,-0.000004
2,Регион,0.002998,0.000176
3,volatility_rel3,0.002934,0.000215
4,momentum_clip,0.002722,-0.000185
5,store_mar_feb_ratio,0.002392,-0.000026
6,momentum_corr,0.002019,0.000064
7,Населенный пункт,0.001953,-0.000067
8,diff1,0.001944,-0.000021
9,volatility_rel6,0.001664,-0.000007


## 10. Reproducibility Check For Candidate 15

`stage2_candidate_15_best_baseline_catboost_residual_cvbest.csv` воспроизводится этим ноутбуком в блоке `Train Final Model And Generate Candidates`.

Точная настройка candidate 15:

`pred = best_base * exp(0.50 * CatBoost_pred)`

без clip.

Этот блок проверяет, что файл создан, имеет правильный формат и совпадает с повторной генерацией из текущих объектов в памяти.

In [12]:
candidate_15_path = SUBMISSIONS_DIR / 'stage2_candidate_15_best_baseline_catboost_residual_cvbest.csv'
assert candidate_15_path.exists()

candidate_15_file = pd.read_csv(candidate_15_path).sort_values('new_id').reset_index(drop=True)
candidate_15_regenerated = pred_rows[['new_id']].copy()
candidate_15_regenerated['rto'] = np.round(base * np.exp(0.50 * pred_log), 2)
candidate_15_regenerated = candidate_15_regenerated.sort_values('new_id').reset_index(drop=True)

checks = {
    'rows': len(candidate_15_file),
    'columns': ', '.join(candidate_15_file.columns),
    'new_id_unique': candidate_15_file['new_id'].is_unique,
    'matches_regenerated': candidate_15_file.equals(candidate_15_regenerated),
    'max_abs_diff': float((candidate_15_file['rto'] - candidate_15_regenerated['rto']).abs().max()),
}
checks

{'rows': 18657,
 'columns': 'new_id, rto',
 'new_id_unique': True,
 'matches_regenerated': True,
 'max_abs_diff': 0.0}

## Итог

Актуальная лучшая модель: best baseline + CatBoost residual.

Подтвержденный leaderboard:

- чистый baseline: около `5.72`;
- `candidate_14`: `5.707`;
- `candidate_15`: `5.58`.

Следующая рабочая гипотеза: подобрать силу residual вокруг `alpha=0.50`, потому что направление уже подтверждено contest.